# YOLO11n Object Detection — SingleNode Multi-GPU Training on Databricks

End-to-end workflow for training [YOLO11n](https://docs.ultralytics.com/models/yolo11/) on [COCO128](https://www.kaggle.com/datasets/ultralytics/coco128) using **single-node multi-GPU distributed training** via [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html) on [Classic GPU Compute](https://docs.databricks.com/en/compute/gpu.html), with [MLflow tracking](https://docs.databricks.com/en/mlflow/index.html) and [Unity Catalog](https://docs.databricks.com/en/data-governance/unity-catalog/index.html) storage.

> **This notebook focuses on distributed training only.** For model serving, endpoint deployment, and inference, see the companion notebook: [`classic-yolo11n-detect-coco128-singleA10`](https://github.com/databricks-industry-solutions/cv-playground/blob/main/projects/ultralytics_databricks_examples/classic-yolo11n-detect-coco128-singleA10.ipynb).

---

> #### ⚠️ Dataset Size & Overfitting
>
> **COCO128 is for demonstration only.** With only 128 images (\~80 train, \~24 val, \~24 test), the model will overfit. For production, use larger datasets (1K+ domain-specific images). This workflow scales directly to larger datasets by updating data paths. See [NuInsSeg](https://github.com/databricks-industry-solutions/cv-playground/tree/main/projects/NuInsSeg) for a real-world example using YOLO instance segmentation on nuclei data.

---

## Workflow
1. **Environment Setup** — Install packages, import libraries, define helper functions
2. **Unity Catalog** — Create schema, volume, configure paths
3. **Data Preparation** — Download COCO128, create train/val/test splits
4. **MLflow Setup** — Configure experiment, infer model signature
5. **Distributed Training** — Train YOLO11n across multiple GPUs with [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html) and MLflow tracking

## Key Features
* **[DistributedDataParallel (DDP)](https://pytorch.org/docs/stable/notes/ddp.html)** via [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html) — Multi-GPU training with [NCCL backend](https://pytorch.org/docs/stable/distributed.html#backends)
* **[Base64 input format](https://en.wikipedia.org/wiki/Base64)** — Universal, network-friendly image encoding
* **[Custom MLflow PyFunc wrapper](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#creating-custom-pyfunc-models)** — Production-ready model packaging
* **[Unity Catalog Volumes](https://docs.databricks.com/en/connect/unity-catalog/volumes.html)** — Managed storage for data, models, and artifacts

## Compute Requirements

This notebook requires a **single-node [Classic GPU cluster](https://docs.databricks.com/en/compute/gpu.html)** with multiple GPUs in single-user access mode. Any multi-GPU node type works (T4, V100, A10, A100, H100) — the key requirement is **at least two GPUs** for distributed data-parallel training. Use [DBR 13.0 ML+](https://docs.databricks.com/en/release-notes/runtime/index.html) (GPU) or later.

| Property | Value |
| --- | --- |
| **Cluster Name** | `HIMSS_CV_SingleNode_MultipeGPU` |
| **Node Type** | `Standard_NC12s_v3` |
| **GPUs** | 2 \~ NVIDIA Tesla V100 (16 GB each) |
| **vCPUs / RAM** | 12 / 224 GB |
| **Topology** | Single Node (0 workers) |
| **DBR** | 17.3 (Spark 17.3.x-scala2.13) |
| **Security Mode** | Single User |
| **Cloud** | Azure |

## Background

This notebook extends the single-GPU approach in [`classic-yolo11n-detect-coco128-singleA10`](https://github.com/databricks-industry-solutions/cv-playground/blob/main/projects/ultralytics_databricks_examples/classic-yolo11n-detect-coco128-singleA10.ipynb) to **multiple GPUs on a single node** using PyTorch [DistributedDataParallel (DDP)](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html). The environment setup, helper functions, data preparation, and MLflow configuration are shared across both notebooks — the key difference is the training cell, which uses [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html) to coordinate across GPUs.

### Distributed Pytorch
[PyTorch distributed](https://pytorch.org/docs/stable/distributed.html) package supports Linux (stable), MacOS (stable), and Windows (prototype). By default for Linux, the Gloo and [NCCL](https://developer.nvidia.com/nccl) backends are built and included in PyTorch distributed (NCCL only when building with CUDA). MPI is an optional backend that can only be included if you build PyTorch from source (e.g. building PyTorch on a host that has MPI installed).

In short ([which backend to use?](https://pytorch.org/docs/stable/distributed.html#which-backend-to-use)):
- use **NCCL** for GPU (**this notebook**)
- use **Gloo** for CPU
- use **MPI** if Gloo won't work for CPU

ref: [PyTorch Distributed Overview](https://pytorch.org/docs/stable/distributed.html) · [Databricks Distributed Training](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/index.html)

### Applying Parallelism To Scale Your Model
Data Parallelism is a widely adopted single-program multiple-data training paradigm where the model is replicated on every process, every model replica computes local gradients for a different set of input data samples, gradients are averaged within the data-parallel communicator group before each optimizer step.

Model Parallelism techniques (or Sharded Data Parallelism) are required when a model doesn't fit in GPU, and can be combined together to form multi-dimensional (N-D) parallelism techniques.

When deciding what parallelism techniques to choose for your model, use these common guidelines:

- Use [DistributedDataParallel (DDP)](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html) (**this notebook**), if your model fits in a single GPU but you want to easily scale up training using multiple GPUs.

  + Use [`torchrun`](https://pytorch.org/docs/stable/elastic/run.html), to launch multiple pytorch processes if you are using more than one node.

  + See also: [Getting Started with DDP](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)

- Use [FullyShardedDataParallel (FSDP)](https://pytorch.org/docs/stable/fsdp.html) when your model cannot fit on one GPU.

  + See also: [Getting Started with FSDP](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)

- Use [Tensor Parallel (TP)](https://pytorch.org/docs/stable/distributed.tensor.parallel.html) and/or [Pipeline Parallel (PP)](https://pytorch.org/docs/stable/pipeline.html) if you reach scaling limitations with FSDP.

  + Try the [Tensor Parallelism Tutorial](https://pytorch.org/tutorials/intermediate/TP_tutorial.html)

  + See also: [TorchTitan end to end example of 3D parallelism](https://github.com/pytorch/torchtitan)

ref: [PyTorch Distributed Overview](https://pytorch.org/tutorials/beginner/dist_overview.html) · [Databricks Distributed Training](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/index.html) · [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html)

## Environment Setup

Install required packages and configure Python environment for YOLO training on Classic GPU compute.

In [0]:
# Core packages
%pip install -U mlflow>=3.0                    # MLflow for experiment tracking and model registry

# Fix OpenCV conflict: Install headless version first, then ultralytics without deps
%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
%pip install opencv-python-headless==4.10.0.84  # Headless OpenCV (no GUI dependencies)

# Install ultralytics without dependencies to prevent opencv-python reinstall
%pip install --no-deps ultralytics==8.3.204

# Manually install ultralytics dependencies (excluding opencv-python)
%pip install matplotlib>=3.3.0 pillow>=7.1.2 pyyaml>=5.3.1 requests>=2.23.0 scipy>=1.4.1 psutil polars ultralytics-thop>=2.0.0

# GPU monitoring (required for MLflow system metrics)
%pip install nvidia-ml-py==13.580.82           # NVIDIA GPU monitoring

# Optional optimization packages
%pip install threadpoolctl==3.1.0              # Controls CPU thread usage
%pip install pyrsmi==0.2.0                     # AMD GPU monitoring (if using AMD GPUs)

dbutils.library.restartPython()

  Using cached mlflow-3.10.1-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.10.1-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached opentelemetry_proto-1.40.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
Using cached mlflow-3.10.1-py3-none-any.whl (10.2 MB)
Using cached mlflow_skinny-3.10.1-py3-none-any.whl (3.0 MB)
Using cache

In [0]:
import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" # Enable synchronous execution for debugging,for debugging purpose, show stack trace immediately. Use it in dev mode.
os.environ["CUDA_LAUNCH_BLOCKING"] = "0" # Reset to asynchronous execution for production 

In [0]:
from ultralytics.utils.checks import check_yolo, check_python, check_latest_pypi_version, check_version, check_requirements

print("check_yolo", check_yolo())
print("check_python", check_python())
print("check_latest_pypi_version", check_latest_pypi_version())
print("check_version", check_version())

Ultralytics 8.3.204 🚀 Python-3.12.3 torch-2.7.0+cu126 CUDA:0 (Tesla V100-PCIE-16GB, 16151MiB)
Setup complete ✅ (12 CPUs, 212.6 GB RAM, 51.6/250.9 GB disk)
check_yolo None
check_python True
check_latest_pypi_version 8.4.21
check_version True


## Helper Functions

Utility functions for the complete YOLO training and deployment workflow:

**Data Management:**
* `download_file()` - Download models and configs to UC Volume
* `download_and_extract_dataset()` - Download and extract COCO128
* `split_dataset()` - Create reproducible train/val/test splits

**MLflow Integration:**
* `infer_model_signature()` - Automatically infer model signature from predictions
* `setup_mlflow_experiment()` - Configure MLflow with system metrics
* `register_yolo_model()` - Register model to Unity Catalog with custom wrapper

**Model Evaluation:**
* `evaluate_model_on_split()` - Evaluate and visualize predictions on data splits

**Custom Wrapper:**
* `YOLOWrapper` - [MLflow PyFunc wrapper](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#creating-custom-pyfunc-models) for YOLO models
  * **Input:** [Base64-encoded](https://en.wikipedia.org/wiki/Base64) images (universal format, works across network boundaries)
  * **Output:** DataFrame with class, confidence, bounding boxes (11 columns)
  * **Purpose:** Enables deployment to [Model Serving endpoints](https://docs.databricks.com/en/machine-learning/model-serving/create-manage-serving-endpoints.html)
  * **Production-ready:** Tested locally before deployment

In [0]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

import os
import shutil
import requests
import zipfile
import io
import random
import yaml
import glob
import json
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import mlflow
from mlflow import MlflowClient
import importlib.metadata


def download_file(url, destination, description="file"):
    """Download a file from URL to destination path."""
    if os.path.exists(destination):
        print(f"[INFO] {description} already exists at: {destination}")
        print(f"   Skipping download")
        return True
    
    print(f"Downloading {description}...")
    print(f"   From: {url}")
    print(f"   To: {destination}")
    
    try:
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            os.makedirs(os.path.dirname(destination), exist_ok=True)
            with open(destination, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"[OK] Downloaded successfully")
            if destination.endswith('.pt'):
                print(f"   Size: {os.path.getsize(destination) / (1024*1024):.2f} MB")
            return True
        else:
            print(f"[ERROR] Download failed with status code: {response.status_code}")
            return False
    except Exception as e:
        print(f"[ERROR] Download failed: {e}")
        return False


def download_and_extract_dataset(download_url, extraction_path):
    """Download and extract a zip dataset."""
    print("Downloading dataset...")
    response = requests.get(download_url)
    
    print("Extracting dataset...")
    z = zipfile.ZipFile(io.BytesIO(response.content))
    z.extractall(extraction_path)
    
    print(f"[OK] Dataset downloaded and extracted to: {extraction_path}")
    return True


def split_dataset(source_images_dir, source_labels_dir, base_images_dir, base_labels_dir,
                  train_ratio=0.625, val_ratio=0.1875, random_seed=42):
    """Split dataset into train/val/test sets with reproducible random seed."""
    print("=" * 60)
    print("DATASET SPLITTING")
    print("=" * 60)
    
    random.seed(random_seed)
    print(f"\nRandom seed: {random_seed}")
    
    test_ratio = 1.0 - train_ratio - val_ratio
    print(f"Split ratios: Train={train_ratio:.1%}, Val={val_ratio:.1%}, Test={test_ratio:.1%}\n")
    
    # Get all images
    all_images = sorted([f for f in os.listdir(source_images_dir) if f.endswith('.jpg')])
    print(f"Total images: {len(all_images)}")
    
    # Shuffle and split
    random.shuffle(all_images)
    train_size = int(len(all_images) * train_ratio)
    val_size = int(len(all_images) * val_ratio)
    
    train_images = all_images[:train_size]
    val_images = all_images[train_size:train_size + val_size]
    test_images = all_images[train_size + val_size:]
    
    print(f"Split sizes: Train={len(train_images)}, Val={len(val_images)}, Test={len(test_images)}\n")
    
    # Create directories
    for split_name in ['train', 'val', 'test']:
        os.makedirs(f"{base_images_dir}/{split_name}", exist_ok=True)
        os.makedirs(f"{base_labels_dir}/{split_name}", exist_ok=True)
    
    # Copy files
    print("Copying files to splits...")
    for split_name, image_list in [('train', train_images), ('val', val_images), ('test', test_images)]:
        print(f"  Processing {split_name} split ({len(image_list)} images)...")
        for img_name in image_list:
            # Copy image
            src_img = os.path.join(source_images_dir, img_name)
            dst_img = os.path.join(base_images_dir, split_name, img_name)
            shutil.copy2(src_img, dst_img)
            
            # Copy label if exists
            label_name = img_name.replace('.jpg', '.txt')
            src_label = os.path.join(source_labels_dir, label_name)
            dst_label = os.path.join(base_labels_dir, split_name, label_name)
            if os.path.exists(src_label):
                shutil.copy2(src_label, dst_label)
        print(f"    [OK] {split_name}: {len(image_list)} images copied")
    
    print(f"\n[OK] Dataset split complete!")
    print("=" * 60)
    return len(train_images), len(val_images), len(test_images)


def infer_model_signature(model_path, sample_image_path):
    """Infer MLflow model signature using actual model predictions."""
    import base64
    
    print("[INFO] Inferring model signature...\n")
    
    # Load YOLO model
    model = YOLO(model_path)
    
    # Read and encode image as base64
    with open(sample_image_path, 'rb') as f:
        image_bytes = f.read()
    image_base64 = base64.b64encode(image_bytes).decode('utf-8')
    
    # Create input example
    input_example = pd.DataFrame({"image_base64": [image_base64]})
    
    # Create YOLOWrapper instance and get predictions to infer output schema
    wrapper = YOLOWrapper()
    
    # Simulate load_context
    class MockContext:
        def __init__(self, model_path):
            self.artifacts = {"yolo_model": model_path}
    
    wrapper.load_context(MockContext(model_path))
    
    # Get output example by running prediction
    output_example = wrapper.predict(None, input_example)
    
    # Use MLflow's infer_signature to automatically create signature
    signature = mlflow.models.infer_signature(input_example, output_example)
    
    print(f"[OK] Model signature inferred successfully!")
    print(f"   Input: DataFrame with 'image_base64' column (base64 string)")
    print(f"   Output: DataFrame with {len(output_example.columns)} columns")
    print(f"   Columns: {', '.join(output_example.columns.tolist())}")
    
    # Optional: Show how to use manual schema (commented out)
    # from mlflow.types.schema import Schema, ColSpec
    # from mlflow.models.signature import ModelSignature
    # input_schema = Schema([ColSpec("string", "image_base64")])
    # output_schema = Schema([
    #     ColSpec("string", "class_name"),
    #     ColSpec("long", "class_num"),
    #     ColSpec("double", "confidence"),
    #     ColSpec("double", "bbox_x1"),
    #     ColSpec("double", "bbox_y1"),
    #     ColSpec("double", "bbox_x2"),
    #     ColSpec("double", "bbox_y2"),
    #     ColSpec("double", "bbox_center_x"),
    #     ColSpec("double", "bbox_center_y"),
    #     ColSpec("double", "bbox_width"),
    #     ColSpec("double", "bbox_height")
    # ])
    # signature = ModelSignature(inputs=input_schema, outputs=output_schema)
    
    return signature, input_example


def setup_mlflow_experiment(notebook_path = None, manual_path = None, expriment_name = "Experiments_YOLO_CoCo"):
    """Setup MLflow experiment with system metrics enabled."""
    if notebook_path:
        notebook_dir = '/'.join(notebook_path.split('/')[:-1])
        experiment_name = f"{notebook_dir}/{expriment_name}"
    elif manual_path:
        experiment_name = f"{manual_path}/{expriment_name}"
    else:
        raise ValueError("Either notebook_path or manual_path must be provided.") 
    
    os.environ['MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING'] = "true"
    os.environ['MLFLOW_EXPERIMENT_NAME'] = experiment_name
    
    mlflow.set_experiment(experiment_name)
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
    
    if 'MLFLOW_RUN_ID' in os.environ:
        del os.environ['MLFLOW_RUN_ID']
    
    print(f"[OK] MLflow experiment initialized: {experiment_name}")
    print(f"   Experiment ID: {experiment_id}")
    print(f"   System metrics: ENABLED")
    return experiment_name, experiment_id


class YOLOWrapper(mlflow.pyfunc.PythonModel):
    """Custom MLflow wrapper for YOLO models using base64-encoded images."""
    
    def load_context(self, context):
        """Load YOLO model from artifacts."""
        from ultralytics import YOLO
        model_path = context.artifacts["yolo_model"]
        self.model = YOLO(model_path, task='detect')
    
    def _format_predictions(self, predictions):
        """Format YOLO prediction results with bounding boxes.
        
        Args:
            predictions: YOLO prediction results from model.predict()
            
        Returns:
            pd.DataFrame with class, confidence, and bounding box coordinates
        """
        import pandas as pd
        
        all_results = []
        for prediction in predictions:
            if prediction.boxes is not None:
                boxes = prediction.boxes
                for i in range(len(boxes)):
                    # Get bounding box coordinates in both formats
                    box_xyxy = boxes.xyxy[i].cpu().numpy()
                    box_xywh = boxes.xywh[i].cpu().numpy()
                    
                    all_results.append({
                        "class_name": prediction.names[int(boxes.cls[i])],
                        "class_num": int(boxes.cls[i]),
                        "confidence": float(boxes.conf[i]),
                        "bbox_x1": float(box_xyxy[0]),
                        "bbox_y1": float(box_xyxy[1]),
                        "bbox_x2": float(box_xyxy[2]),
                        "bbox_y2": float(box_xyxy[3]),
                        "bbox_center_x": float(box_xywh[0]),
                        "bbox_center_y": float(box_xywh[1]),
                        "bbox_width": float(box_xywh[2]),
                        "bbox_height": float(box_xywh[3])
                    })
        
        return pd.DataFrame(all_results)
    
    def predict(self, context, model_input):
        """Run YOLO prediction on base64-encoded images.
        
        Args:
            context: MLflow context
            model_input: DataFrame with 'image_base64' column (base64-encoded images)
            
        Returns:
            pd.DataFrame with detection results including bounding boxes
        """
        import pandas as pd
        import base64
        from PIL import Image
        import io
        import numpy as np
        
        if not isinstance(model_input, pd.DataFrame):
            raise ValueError("Input must be a DataFrame with 'image_base64' column")
        
        if 'image_base64' not in model_input.columns:
            raise ValueError("DataFrame must contain 'image_base64' column with base64-encoded images")
        
        # Process base64-encoded images
        all_predictions = []
        for image_base64 in model_input['image_base64'].tolist():
            # Decode base64 to image
            image_bytes = base64.b64decode(image_base64)
            image = Image.open(io.BytesIO(image_bytes))
            image_array = np.array(image)
            
            # Run prediction
            predictions = self.model.predict(image_array, verbose=False)
            all_predictions.extend(predictions)
        
        return self._format_predictions(all_predictions)


def register_yolo_model(run_id, model_path, catalog_name, schema_name, model_name,
                       signature=None, input_example=None, data_yaml_path=None):
    """Register YOLO model to Unity Catalog with custom wrapper."""
    registered_model_name = f"{catalog_name}.{schema_name}.{model_name}"
    ultralytics_version = importlib.metadata.version('ultralytics')
    cloudpickle_version = importlib.metadata.version('cloudpickle')
    
    print(f"\n[INFO] Registering model to Unity Catalog...")
    print(f"   Model name: {registered_model_name}")
    print(f"   Using custom YOLO wrapper (base64 input, bbox output)")
    print(f"   Pinning CloudPickle version: {cloudpickle_version}")
    
    with mlflow.start_run(run_id=run_id):
        if data_yaml_path:
            mlflow.log_artifact(data_yaml_path, "input_data")
        
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=YOLOWrapper(),
            artifacts={"yolo_model": model_path},
            signature=signature,
            input_example=input_example,
            registered_model_name=registered_model_name,
            pip_requirements=[
                f"ultralytics=={ultralytics_version}",
                f"cloudpickle=={cloudpickle_version}",
                "torch", 
                "torchvision", 
                "pillow", 
                "numpy"
            ]
        )
    
    print(f"   [OK] Model registered: {registered_model_name}")
    return registered_model_name


def evaluate_model_on_split(model, image_dir, split_name, output_dir, run_id, 
                           registered_model_name, organized_run_name, num_samples=3):
    """Evaluate model on a dataset split and save results."""
    print("=" * 60)
    print(f"{split_name.upper()} SET EVALUATION")
    print("=" * 60)
    
    os.makedirs(output_dir, exist_ok=True)
    images = glob.glob(f"{image_dir}/*.jpg")
    
    if not images:
        print(f"[WARNING] No {split_name} images found")
        return
    
    print(f"\n{split_name.capitalize()} set: {len(images)} images\n")
    
    # Visualize sample predictions
    sample_images = images[:num_samples]
    fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))
    if len(sample_images) == 1:
        axes = [axes]
    
    results = []
    for i, img_path in enumerate(sample_images):
        print(f"Sample {i+1}/{len(sample_images)}: {img_path.split('/')[-1]}")
        predictions = model.predict(img_path, verbose=False)
        
        if len(predictions) > 0:
            result = predictions[0]
            annotated_img = result.plot()
            axes[i].imshow(annotated_img)
            axes[i].axis('off')
            
            if result.boxes is not None:
                num_detections = len(result.boxes)
                axes[i].set_title(f"{img_path.split('/')[-1]}\n{num_detections} objects", fontsize=10)
                print(f"   [OK] Detections: {num_detections} objects")
                
                img_results = {
                    "image": img_path.split('/')[-1],
                    "num_detections": num_detections,
                    "detections": []
                }
                
                for j in range(min(num_detections, 3)):
                    class_name = result.names[int(result.boxes.cls[j])]
                    confidence = float(result.boxes.conf[j])
                    print(f"      - {class_name}: {confidence:.3f}")
                    img_results["detections"].append({
                        "class_name": class_name,
                        "confidence": confidence
                    })
                results.append(img_results)
        print()
    
    plt.tight_layout()
    plt.suptitle(f"{split_name.capitalize()} Set Predictions - Run {run_id[:8]}", fontsize=14, y=1.02)
    
    plot_path = os.path.join(output_dir, f"{split_name}_predictions.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"[OK] Plot saved to: {plot_path}")
    plt.show()
    
    # Save results JSON
    json_path = os.path.join(output_dir, f"{split_name}_results.json")
    with open(json_path, 'w') as f:
        json.dump({
            "run_id": run_id,
            "registered_model": registered_model_name,
            "timestamp": organized_run_name.split('_run_')[0],
            "num_images": len(images),
            "sample_results": results
        }, f, indent=2)
    print(f"[OK] Results saved to: {json_path}")
    
    # Log to MLflow
    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifact(plot_path, split_name)
        mlflow.log_artifact(json_path, split_name)
    
    print(f"\n[OK] {split_name.upper()} SET EVALUATION COMPLETE")
    print("=" * 60)


print("[OK] Helper functions loaded successfully")

[OK] Helper functions loaded successfully


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1ae40fbd-4d55-4d65-af15-38d493f2af2e/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Unity Catalog Configuration

Configure widgets, create schema/volume, and set up project paths.

In [0]:
# Clear existing widgets
dbutils.widgets.removeAll()

In [0]:
# Define widgets for catalog, schema, volume, and model name
dbutils.widgets.text("catalog_name", "mmt", "Catalog Name") ## update to use your catalog name
dbutils.widgets.text("schema_name", "cv", "Schema Name") ## update to use your schema name
dbutils.widgets.text("volume_name", "yolo_classic", "Volume Name") 
dbutils.widgets.text("model_name", "yolo11n_coco128_classic", "Model Name")

# Get widget values
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
model_name = dbutils.widgets.get("model_name")

print(f"[Configuration]")
print(f"   Catalog: {catalog_name}")
print(f"   Schema: {schema_name}")
print(f"   Volume: {volume_name}")
print(f"   Model: {model_name}")

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"\n[OK] Schema: {catalog_name}.{schema_name}")

# Create volume for persistent storage
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
print(f"[OK] Volume: {catalog_name}.{schema_name}.{volume_name}")

[Configuration]
   Catalog: mmt
   Schema: cv
   Volume: yolo_classic
   Model: yolo11n_coco128_classic

[OK] Schema: mmt.cv
[OK] Volume: mmt.cv.yolo_classic


In [0]:
# Construct volume path from parameters
project_location = f'/Volumes/{catalog_name}/{schema_name}/{volume_name}/'
print(f"Using Unity Catalog Volume: {catalog_name}.{schema_name}.{volume_name}")
print(f"Volume path: {project_location}")

# Create subdirectories in the volume
os.makedirs(f'{project_location}runs/', exist_ok=True)       # Training runs (organized by task/model/dataset)
os.makedirs(f'{project_location}data/', exist_ok=True)       # Dataset storage
os.makedirs(f'{project_location}raw_model/', exist_ok=True)  # Pretrained models

# Ephemeral /tmp/ location for faster I/O during training
tmp_project_location = "/tmp/training_results/"
os.makedirs(tmp_project_location, exist_ok=True)

print(f"\n[OK] Project directories created:")
print(f"   Runs: {project_location}runs/")
print(f"   Data: {project_location}data/")
print(f"   Raw models: {project_location}raw_model/")
print(f"   Temp (training): {tmp_project_location}  # Ephemeral, fast I/O")

Using Unity Catalog Volume: mmt.cv.yolo_classic
Volume path: /Volumes/mmt/cv/yolo_classic/

[OK] Project directories created:
   Runs: /Volumes/mmt/cv/yolo_classic/runs/
   Data: /Volumes/mmt/cv/yolo_classic/data/
   Raw models: /Volumes/mmt/cv/yolo_classic/raw_model/
   Temp (training): /tmp/training_results/  # Ephemeral, fast I/O


## Project Folder Structure

Unity Catalog Volume organization:

```
/Volumes/{catalog}/{schema}/{volume}/
├── data/
│   ├── coco128.yaml                    # Dataset configuration
│   └── coco128/
│       ├── images/
│       │   ├── train2017/              # Original 128 images (from zip)
│       │   ├── train/  val/  test/     # Custom splits (80/24/24)
│       └── labels/
│           ├── train2017/              # Original labels
│           └── train/  val/  test/     # Split labels
│
├── raw_model/
│   └── yolo11n.pt                      # Pretrained YOLO11n weights
│
└── runs/
    └── {task}_{model}_{dataset}_{timestamp}_run_{mlflow_run_id}/
        └── train/                      # Training outputs
            ├── weights/ (best.pt, last.pt)
            └── results.csv, confusion_matrix.png
```

**Run Naming:** `detection_yolo11n_coco128_20260120_143052_run_{mlflow_run_id}`
* Includes task, model, dataset, timestamp, and MLflow run ID for easy identification

> **Evaluation & serving folders** (`validation_samples/`, `test_samples/`) are created by the companion [`singleA10`](https://github.com/databricks-industry-solutions/cv-playground/blob/main/projects/ultralytics_databricks_examples/classic-yolo11n-detect-coco128-singleA10.ipynb) notebook, which covers model evaluation and deployment.

## Data Preparation

Download pretrained YOLO11n weights and COCO128 dataset to Unity Catalog Volume.

In [0]:
import yaml

print("=" * 60)
print("DOWNLOADING ASSETS")
print("=" * 60)

# ============================================================
# 1. Download Pretrained YOLO11n Model
# ============================================================
print("\n[1/2] Downloading pretrained YOLO11n model...")
model_path = f"{project_location}raw_model/yolo11n.pt"
model_url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt"

download_file(model_url, model_path, "YOLO11n model")
print(f"[OK] Pretrained model ready at: {model_path}")

# ============================================================
# 2. Download and Prepare COCO128 Dataset
# ============================================================
print("\n[2/2] Downloading COCO128 dataset...")

# Create data directory in UC Volume
os.makedirs(f'{project_location}data/coco128', exist_ok=True)

# Download config directly to UC Volume
config_url = "https://github.com/ultralytics/ultralytics/raw/main/ultralytics/cfg/datasets/coco128.yaml"
config_path = f"{project_location}data/coco128.yaml"

download_file(config_url, config_path, "COCO128 config")

# Load and update configuration
with open(config_path, 'r') as f:
    data = yaml.safe_load(f)

print(f"\n[Dataset Configuration]")
print(f"   Dataset: {data.get('path', 'coco128')}")
print(f"   Classes: {data.get('nc', 'unknown')}")
print(f"   Download URL: {data.get('download', 'N/A')}")

# Update paths for Unity Catalog Volume
data['path'] = f"{project_location}data/coco128"

# Check if dataset already exists before downloading
dataset_images_dir = f"{project_location}data/coco128/images/train2017"
if os.path.exists(dataset_images_dir) and len(os.listdir(dataset_images_dir)) > 0:
    print(f"\n[INFO] Dataset already exists at: {dataset_images_dir}")
    print(f"   Found {len(os.listdir(dataset_images_dir))} images")
    print(f"   Skipping download")
else:
    # Download and extract dataset only if it doesn't exist
    extraction_path = f"{project_location}data"
    download_and_extract_dataset(data['download'], extraction_path)

# Save updated configuration to UC Volume
data_yaml_path = f"{project_location}data/coco128.yaml"
with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"\n[OK] Dataset configuration saved to UC Volume: {data_yaml_path}")
print(f"   All dataset files in: {project_location}data/coco128/")

print("\n" + "=" * 60)
print("[OK] ALL ASSETS DOWNLOADED")
print("=" * 60)

DOWNLOADING ASSETS

[1/2] Downloading pretrained YOLO11n model...
[INFO] YOLO11n model already exists at: /Volumes/mmt/cv/yolo_classic/raw_model/yolo11n.pt
   Skipping download
[OK] Pretrained model ready at: /Volumes/mmt/cv/yolo_classic/raw_model/yolo11n.pt

[2/2] Downloading COCO128 dataset...
[INFO] COCO128 config already exists at: /Volumes/mmt/cv/yolo_classic/data/coco128.yaml
   Skipping download

[Dataset Configuration]
   Dataset: /Volumes/mmt/cv/yolo_classic/data/coco128
   Classes: unknown
   Download URL: https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip

[INFO] Dataset already exists at: /Volumes/mmt/cv/yolo_classic/data/coco128/images/train2017
   Found 128 images
   Skipping download

[OK] Dataset configuration saved to UC Volume: /Volumes/mmt/cv/yolo_classic/data/coco128.yaml
   All dataset files in: /Volumes/mmt/cv/yolo_classic/data/coco128/

[OK] ALL ASSETS DOWNLOADED


## MLflow Configuration

Infer model signature, enable system metrics, and configure experiment tracking.

In [0]:
# Infer model signature from sample prediction
# This defines the input/output schema for the serving endpoint
# Input: base64-encoded images
# Output: class, confidence, bounding boxes (11 columns)

model_path = f"{project_location}raw_model/yolo11n.pt"

# Find a sample image from training set
sample_images = glob.glob(f"{project_location}data/coco128/images/train2017/*.jpg")

if sample_images:
    signature, input_example = infer_model_signature(model_path, sample_images[0])
    print(f"\n[OK] Signature and input example ready for model registration")
else:
    print("[WARNING] No sample images found. Run dataset preparation first.")
    signature = None
    input_example = None

[INFO] Inferring model signature...

[OK] Model signature inferred successfully!
   Input: DataFrame with 'image_base64' column (base64 string)
   Output: DataFrame with 11 columns
   Columns: class_name, class_num, confidence, bbox_x1, bbox_y1, bbox_x2, bbox_y2, bbox_center_x, bbox_center_y, bbox_width, bbox_height

[OK] Signature and input example ready for model registration


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1ae40fbd-4d55-4d65-af15-38d493f2af2e/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [0]:
signature, input_example

(inputs: 
   ['image_base64': string (required)]
 outputs: 
   ['class_name': string (required), 'class_num': long (required), 'confidence': double (required), 'bbox_x1': double (required), 'bbox_y1': double (required), 'bbox_x2': double (required), 'bbox_y2': double (required), 'bbox_center_x': double (required), 'bbox_center_y': double (required), 'bbox_width': double (required), 'bbox_height': double (required)]
 params: 
   None,
                                         image_base64
 0  /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...)

In [0]:
# Configure YOLO to use MLflow
from ultralytics import settings
settings.update({"mlflow": True})

# Enable MLflow autologging for system metrics
mlflow.autolog(disable=False)

print(f"\n[MLflow Configuration]")
print(f"   YOLO MLflow integration: Enabled")
print(f"   MLflow autologging: Enabled")
print(f"   System metrics: Enabled")

2026/03/10 09:27:05 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/03/10 09:27:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/03/10 09:27:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.



[MLflow Configuration]
   YOLO MLflow integration: Enabled
   MLflow autologging: Enabled
   System metrics: Enabled


In [0]:
# setup mlflow tracking and registry
from mlflow.exceptions import RestException
import mlflow 
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# notebook cred is short-lived; for PROD, please use SP.
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
created = w.tokens.create(comment="temp token for notebook use", lifetime_seconds=3600)
DATABRICKS_TOKEN = created.token_value
HOST = w.config.host
os.environ["DATABRICKS_HOST"] = HOST
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
print(f"\n[MLflow Configuration]")
print(f"   Databricks Host: {HOST}")
print(f"   Databricks Token: {DATABRICKS_TOKEN}")

#: Setup MLflow experiment with system metrics,
# NOTE: if notebook is not in a git repo, this will throw an error
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
try:
    experiment_name, experiment_id = setup_mlflow_experiment(notebook_path = notebook_path)
except RestException as e:
    print(f"[ERROR] Unable to setup MLflow experiment: {e}")

#: now setup manuall according to your workspace folder (dont put inside a git repo folder)
experiment_name, experiment_id = setup_mlflow_experiment(manual_path = "/Workspace/Users/yang.yang@databricks.com/")

#: UC model name
registered_model_name = f"{catalog_name}.{schema_name}.{model_name}"


print(f"\n[Ready for Training]")
print(f"   Experiment: {experiment_name}")
print(f"   Experiment ID: {experiment_id}")
print(f"   Registered Model: {registered_model_name}")


[MLflow Configuration]
   Databricks Host: https://eastus2.azuredatabricks.net
   Databricks Token: [REDACTED]
[OK] MLflow experiment initialized: /Users/may.merkletan@databricks.com/TEST/(Clone) ultralytics_databricks_examples/Experiments_YOLO_CoCo
   Experiment ID: 3475686035986609
   System metrics: ENABLED
[OK] MLflow experiment initialized: /Workspace/Users/yang.yang@databricks.com//Experiments_YOLO_CoCo
   Experiment ID: 2278904696588390
   System metrics: ENABLED

[Ready for Training]
   Experiment: /Workspace/Users/yang.yang@databricks.com//Experiments_YOLO_CoCo
   Experiment ID: 2278904696588390
   Registered Model: mmt.cv.yolo11n_coco128_classic


## Model Training

Train YOLO11n using `TorchDistributor` for single-node multi-GPU distributed training with MLflow tracking.

### SingleNode-MultiGPU with TorchDistributor

The training function is distributed across all available GPUs using `TorchDistributor(num_processes=num_gpus, local_mode=True, use_gpu=True)`. Each GPU runs a replica of the model with DDP and the NCCL backend. Only the global rank 0 process logs artifacts and registers the model to MLflow.

In [0]:
import torch
from pyspark.ml.torch.distributor import TorchDistributor

settings.update({"mlflow":True}) # if you do want to autolog.
mlflow.autolog(disable = False)


def train_fn(device_list = [0,1,2,3], active_run_id = None):

    try: 
        from ultralytics import YOLO
        import torch
        import mlflow
        import torch.distributed as dist
        from ultralytics import settings
        from mlflow.types.schema import Schema, ColSpec
        from mlflow.models.signature import ModelSignature

        # MLflow env vars (DATABRICKS_HOST, DATABRICKS_TOKEN, etc.) are inherited
        # from the parent process (set in cell 23). No need to set them here since
        # TorchDistributor(local_mode=True) spawns local subprocesses that share
        # the parent's environment. For PROD, use a service principal instead.
        experiment = mlflow.set_experiment(experiment_name)
        
        #
        with mlflow.start_run(run_id=active_run_id) as run:
            model = YOLO(f"{model_path}")
            model.train(
                batch=8,
                device=device_list,
                data=f"{data_yaml_path}", # ref: https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/coco8.yaml
                epochs=5,
                project=f'{tmp_project_location}',
                exist_ok=True,
                fliplr=1,
                flipud=1,
                perspective=0.001,
                degrees=.45
            )

        # active_run_id = mlflow.last_active_run().info.run_id
        # print("For YOLO autologging, active_run_id is: ", active_run_id)

        # # after training is done.
        # if not dist.is_initialized():
        #   # import torch.distributed as dist
        #   dist.init_process_group("nccl")

        local_rank = int(os.environ["LOCAL_RANK"])
        global_rank = int(os.environ["RANK"])
        
        if global_rank == 0:

            active_run_id = mlflow.last_active_run().info.run_id
            print("For YOLO autologging, active_run_id is: ", active_run_id)

            # Get the list of runs in the experiment
            runs = mlflow.search_runs(experiment_names=[experiment_name], order_by=["start_time DESC"], max_results=1)

            # Extract the latest run_id
            if not runs.empty:
                latest_run_id = runs.iloc[0].run_id
                print(f"Latest run_id: {latest_run_id}")
            else:
                print("No runs found in the experiment.")


            with mlflow.start_run(run_id=latest_run_id) as run:
                mlflow.log_artifact(f"{data_yaml_path}", "input_data_yaml")
                mlflow.log_dict(data, "data.yaml")
                mlflow.log_params({"rank":global_rank})
                yolo_wrapper = YOLOWrapper()
                mlflow.pyfunc.log_model(
                    artifact_path="model",
                    artifacts={'yolo_model': str(model.trainer.save_dir), "best_point": str(model.trainer.best)},
                    python_model=yolo_wrapper,
                    input_example=input_example,
                    registered_model_name=registered_model_name,
                    signature=signature)

        return "finished" # can return any picklable object
    
    finally:
        # clean up
        if dist.is_initialized():
            dist.destroy_process_group()


if __name__ == "__main__":
    mlflow.end_run()
   
    mlflow.set_experiment(experiment_name)
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

    # Reset MLFLOW_RUN_ID, so we dont bump into the wrong one.
    if 'MLFLOW_RUN_ID' in os.environ:
        del os.environ['MLFLOW_RUN_ID']

    num_gpus = torch.cuda.device_count()
    device_list = list(range(num_gpus))
    print("num_gpus:", num_gpus)
    print("device_list:", device_list)

    with mlflow.start_run(experiment_id=experiment_id) as run:
        active_run_id = mlflow.last_active_run().info.run_id
        active_run_name = mlflow.last_active_run().info.run_name

        print("For master triggering run, active_run_id is: ", active_run_id)
        print("For master triggering run, active_run_name is: ", active_run_name)
        print(f"For master triggering run, active_run_id is: '{active_run_id}' and active_run_name is: '{active_run_name}'.")
        print(f"All worker runs will be logged into the same run id '{active_run_id}' and name '{active_run_name}'.")

        distributor = TorchDistributor(num_processes=num_gpus, local_mode=True, use_gpu=True)      
        distributor.run(train_fn, device_list = device_list, active_run_id = active_run_id)

    # Enhanced run naming: {task}_{model}_{dataset}_{datetime}_run_{mlflow_run_id}
    # Copy training results from /tmp to UC Volume with descriptive folder name
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_task = "detection"
    model_arch = "yolo11n"
    dataset_name = "coco128"
    organized_run_name = f"{model_task}_{model_arch}_{dataset_name}_{timestamp}_run_{active_run_id}"
    volume_run_dir = os.path.join(project_location, "runs", organized_run_name)

    shutil.copytree(tmp_project_location, volume_run_dir, dirs_exist_ok=True)
    print(f"\n[OK] Training results copied to: {volume_run_dir}")

2026/03/10 09:27:26 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/03/10 09:27:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/03/10 09:27:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.


num_gpus: 2
device_list: [0, 1]


2026/03/10 09:27:27 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


  Error =>  [Errno 2] No such file or directory: 'rocminfo'
For master triggering run, active_run_id is:  2ea8e77abd6a46b48b38407c1bfa1be1
For master triggering run, active_run_name is:  powerful-wren-223
For master triggering run, active_run_id is: '2ea8e77abd6a46b48b38407c1bfa1be1' and active_run_name is: 'powerful-wren-223'.
All worker runs will be logged into the same run id '2ea8e77abd6a46b48b38407c1bfa1be1' and name 'powerful-wren-223'.


Started local training with 2 processes


Tue Mar 10 09:27:37 2026 Connection to spark using Py4J from PID  12352
Tue Mar 10 09:27:37 2026 Initialized gateway on port 39723
Tue Mar 10 09:27:37 2026 Connection to spark using Py4J from PID  12351
Tue Mar 10 09:27:37 2026 Initialized gateway on port 39843
Tue Mar 10 09:27:37 2026 Connected to spark using Py4J in {spark_mode} mode.
Tue Mar 10 09:27:37 2026 Connected to spark using Py4J in {spark_mode} mode.
  Error =>  [Errno 2] No such file or directory: 'rocminfo'
  Error =>  [Errno 2] No such file or directory: 'rocminfo'
2026/03/10 09:27:38 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/10 09:27:38 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.204 🚀 Python-3.12.3 torch-2.7.0+cu126 CUDA:0 (Tesla V100-PCIE-16GB, 16151MiB)
                                                      CUD

Finished local training with 2 processes
2026/03/10 09:31:05 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/10 09:31:05 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Uploading artifacts: 100%|██████████| 33/33 [00:01<00:00, 31.81it/s]
🔗 Created version '3' of model 'mmt.cv.yolo11n_coco128_classic': https://adb-984752964297111.11.azuredatabricks.net/explore/data/models/mmt/cv/yolo11n_coco128_classic/version/3
🏃 View run powerful-wren-223 at: https://adb-984752964297111.11.azuredatabricks.net/ml/experiments/2278904696588390/runs/2ea8e77abd6a46b48b38407c1bfa1be1
🧪 View experiment at: https://adb-984752964297111.11.azuredatabricks.net/ml/experiments/2278904696588390
2026/03/10 09:31:03 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/10 09:31:03 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!

[OK] Training results copied to: /Volumes/mmt/cv/yolo_classic/runs/detection_yolo11n_coco128_20260310_093105_run_2ea8e77abd6a46b48b38407c1bfa1be1


## Conclusion

This notebook demonstrates **single-node multi-GPU distributed training** of YOLO11n on Databricks:

### What This Notebook Covers

* **Distributed Training**: Leveraged [`TorchDistributor`](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html) with [DDP](https://pytorch.org/docs/stable/notes/ddp.html) to efficiently utilize multiple GPUs, accelerating model training
* **MLflow Integration**: Comprehensive experiment tracking with automatic logging of metrics, parameters, artifacts, and system metrics
* **Unity Catalog Storage**: Organized project assets (datasets, models, runs) in [Unity Catalog Volumes](https://docs.databricks.com/en/connect/unity-catalog/volumes.html) for reproducibility and governance
* **Model Packaging**: Custom `YOLOWrapper` for [MLflow model serving](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#creating-custom-pyfunc-models) with proper input/output signatures

### Training Configuration

* **Model**: [YOLO11n](https://docs.ultralytics.com/models/yolo11/) (nano variant for fast inference)
* **Dataset**: COCO128 (128 images, 80 classes)
* **Training**: 5 epochs with data augmentation (flip, perspective, rotation)
* **Infrastructure**: Multi-GPU training with [NCCL backend](https://pytorch.org/docs/stable/distributed.html#backends) via `TorchDistributor`

### Not Covered in This Notebook

The following steps are implemented in the companion notebook [`classic-yolo11n-detect-coco128-singleA10`](https://github.com/databricks-industry-solutions/cv-playground/blob/main/projects/ultralytics_databricks_examples/classic-yolo11n-detect-coco128-singleA10.ipynb):

* **Model Evaluation** — Validation and test set evaluation using native YOLO format
* **Registered Model Testing** — Local inference test of the MLflow-registered model using base64-encoded images (serving format)
* **Model Serving** — Deployment to a Databricks [Model Serving endpoint](https://docs.databricks.com/en/machine-learning/model-serving/create-manage-serving-endpoints.html) with [AI Gateway inference tables](https://docs.databricks.com/en/ai-gateway/inference-tables/)

### Next Steps

1. **Test & Deploy**: Follow the companion notebook above to validate the registered model and deploy to a serving endpoint
2. **Hyperparameter Tuning**: Experiment with batch sizes, learning rates, and augmentation strategies
3. **Production Deployment**: Scale to larger datasets and production workloads

---

**Distributed Training Complete** ✓